<a href="https://colab.research.google.com/github/MdWasifAli07/softcom-project-/blob/main/Final_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "scikit-learn>=1.9,<2" "matplotlib>=3.9,<4"

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay,
)

PROJECT_DIR = Path('/content/drive/MyDrive/softcom-prompt-injection')
ART_DIR = PROJECT_DIR / 'artifacts'

Load the combined test predictions

In [ ]:
tst = pd.read_csv(ART_DIR / 'ensemble_test.csv')
y_test = tst['label'].to_numpy(dtype=int)
print('Test shape:', tst.shape)
tst.head()

Metrics per layer + combination strategy

In [ ]:
def predict_at_threshold(prob, threshold=0.5):
    return (np.asarray(prob) >= threshold).astype(int)

def metric_dict(y_true, prob, threshold=0.5):
    pred = predict_at_threshold(prob, threshold)
    return {
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'pr_auc': float(average_precision_score(y_true, prob)),
    }

SCORE_COLUMNS = {
    'tfidf_xgboost': 'tfidf_xgb_prob',
    'deberta': 'deberta_prob',
    'weighted_vote': 'weighted_vote_prob',
    'logistic_meta': 'meta_prob',
}

results = {}
for name, col in SCORE_COLUMNS.items():
    prob = tst[col].to_numpy()
    results[name] = metric_dict(y_test, prob)
    print(f'\n=== {name} ===')
    for k, v in results[name].items():
        print(f'{k:10s}: {v:.4f}')
    print(classification_report(
        y_test, predict_at_threshold(prob), target_names=['BENIGN', 'ATTACK'], zero_division=0
    ))

(ART_DIR / 'evaluation_metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print('\nSaved:', ART_DIR / 'evaluation_metrics.json')

Side-by-side comparison, sorted by recall on the attack class

In [ ]:
comparison = pd.DataFrame(results).T[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']]
comparison = comparison.sort_values('recall', ascending=False)
comparison